# Automated Technical File demo

This notebook shows how a raw JSONL robot-run ledger becomes a Markdown corpus page for downstream Mistral retrieval and Q&A.

**Cells 1-2:** Ledger to Markdown conversion (CB-04/CB-05)
**Cells 3-4:** Document Library + Cited Q&A (CB-06 - Misty)

In [ ]:
from pathlib import Path
import subprocess
import sys

root = Path.cwd()
if not (root / "tools" / "ledger_to_md.py").exists():
    root = root / "third_party" / "automated-technical-file"

ledger = root / "artifacts" / "ledger" / "sample_events.jsonl"
wiki_page = root / "artifacts" / "wiki" / "sample_run_summary.md"

subprocess.run(
    [
        sys.executable,
        str(root / "tools" / "ledger_to_md.py"),
        "--input",
        str(ledger),
        "--output",
        str(wiki_page),
        "--title",
        "Sample Automated Technical File Run",
    ],
    check=True,
)

print(wiki_page.read_text(encoding="utf-8")[:1200])

In [ ]:
# Cell 3: Build Mistral Document Library over wiki corpus MD files

import os
from pathlib import Path
from typing import Optional

# Configuration
MISTRAL_API_KEY = os.environ.get('MISTRAL_API_KEY')
BACKEND = os.environ.get('BACKEND', 'auto').lower()

# Model IDs from CB-01 SPEC_models.md
MODEL_HOSTED = 'mistral-large-latest'
MODEL_LOCAL = 'ministral-3b-latest'

# Wiki corpus file paths (relative to automated-technical-file dir)
WIKI_CORPUS_FILES = [
    'notebooks/Overview.md',
    'notebooks/Topics/Compliance.md',
    'notebooks/Topics/BiddingRules.md',
    'notebooks/Subsystems/HardwareInterface.md'
]

try:
    from mistralai import Mistral
except ImportError:
    print('ERROR: mistralai package not installed. Run: pip install mistralai')
    LIBRARY_ID = None
    DOCUMENT_IDS = []
else:
    def create_document_library(api_key, library_name='RobotRoss ATF Wiki'):
        client = Mistral(api_key=api_key)
        library = client.beta.libraries.create(
            name=library_name,
            description='Wiki corpus for RobotRoss Automated Technical File'
        )
        print(f'Created library: {library.name} (ID: {library.id})')
        uploaded_docs = []
        base_path = Path.cwd() / 'third_party' / 'automated-technical-file'
        for file_path in WIKI_CORPUS_FILES:
            full_path = base_path / file_path
            if not full_path.exists():
                print(f'WARNING: File not found: {full_path}')
                continue
            with open(full_path, 'rb') as f:
                doc = client.beta.libraries.documents.upload(
                    library_id=library.id,
                    file={'file_name': file_path, 'content': f.read()}
                )
            uploaded_docs.append(doc)
            print(f'Uploaded: {file_path} (doc ID: {doc.id})')
        return {'library': library, 'documents': uploaded_docs}
    
    if BACKEND == 'hosted' and MISTRAL_API_KEY:
        library_data = create_document_library(MISTRAL_API_KEY)
        LIBRARY_ID = library_data['library'].id
        DOCUMENT_IDS = [d.id for d in library_data['documents']]
        print(f'Library created with {len(DOCUMENT_IDS)} documents')
    elif BACKEND == 'local':
        print('Local backend: Document Library uses in-notebook index')
        LIBRARY_ID = None
        DOCUMENT_IDS = []
    else:
        print('Auto backend: Try hosted first, fall back to local')
        LIBRARY_ID = None
        DOCUMENT_IDS = []

In [ ]:
# Cell 4: Cited Q&A using Agents API + native Citations

def create_cited_qa_agent(api_key, library_id=None):
    client = Mistral(api_key=api_key)
    tools = []
    if library_id:
        tools.append({'type': 'document_library', 'document_library': {'library_ids': [library_id]}})
    agent = client.agents.create(
        name='ATF Q&A Assistant',
        model=MODEL_HOSTED,
        description='Answer questions about RobotRoss ATF with cited sources',
        instructions='You are a helpful assistant. Always ground answers in documents. Include citations.',
        tools=tools
    )
    return agent

def ask_with_citations(agent, question):
    client = Mistral(api_key=MISTRAL_API_KEY)
    response = client.agents.completions.create(
        agent_id=agent.id,
        messages=[{'role': 'user', 'content': question}]
    )
    answer_parts = []
    sources = []
    for chunk in response.choices[0].message.content:
        if hasattr(chunk, 'type'):
            if chunk.type == 'text':
                answer_parts.append(chunk.text)
            elif chunk.type == 'tool_reference':
                sources.append(chunk.tool_reference.reference)
    answer = ''.join(answer_parts)
    if sources:
        answer += '\n\n---\n**Sources:**\n' + '\n'.join(f'- {s}' for s in sources)
    return answer

print('=' * 60)
print('Example: Cited Q&A with Mistral Document Library')
print('=' * 60)

if BACKEND == 'hosted' and MISTRAL_API_KEY and LIBRARY_ID:
    agent = create_cited_qa_agent(MISTRAL_API_KEY, LIBRARY_ID)
    question = "What is the bidding rule for the Wall of Fame?"
    answer = ask_with_citations(agent, question)
    print(f'Q: {question}')
    print(f'A: {answer}')
elif BACKEND == 'local':
    print('Local backend: Use atf_qa.py from ATF/tools/')
    print('Example: python3 ATF/tools/atf_qa.py "What is the bidding rule?"')
else:
    print('Set BACKEND=hosted and MISTRAL_API_KEY for Document Library')
    print('Or BACKEND=local for atf_qa.py corpus approach')